# Train and Validate
* Train a model across the specified training sites and sampling approach
* Validate the model at the excluded site

## Todo
* Pull out probabilities using model.predict_proba(observations_to_predict)

In [ ]:
import geopandas
import pathlib
import rioxarray
import numpy
import dask.distributed
import pandas
import rasterio

module_path = pathlib.Path.cwd().parent / 'scripts'
import sys
if str(module_path) not in sys.path:
    sys.path.append(str(module_path))
import utils
import sentinel2
import training
import sampling

%load_ext autoreload
%autoreload 2

# Values to edit

In [ ]:
sample_method = "sampling_2"
method_2_threshold = .98 # .98 .99 .95

In [ ]:
# View the amount of training data for the selected sampling apporach
samples_per_site = pandas.read_csv(utils.get_samples_summary_file_path(sample_method, method_2_threshold))
samples_per_site

In [ ]:
all_training_sites =  ["CatlinsRiverMouth", "CatlinsLake", "Duvauchelle",
                       "Robinsons", "Childrens", "Takamatua", "Ihutai", "Purau"] 

In [ ]:
test_site = "Robinsons"
training_sites = ["Duvauchelle"] # all_training_sites.copy() # ["Duvauchelle", "Ihutai", "Purau"]
model_file = util.get_model_path(f"test_exclude_{test_site}.joblib")

prediction_file = util.get_prediction_path(f"{test_site}_prediction_{model_file.stem}.nc")

# Ensure not training on test site
if test_site in training_sites:
    training_sites.remove(test_site)


In [ ]:
# Mappings of Satellite classes to consider from UAV classes
uav_classes_to_ignore = ['Shadow', 'Glare']
satellite_classes = {'Seagrass': 1, 'Mixed': 2, 'Unvegetated': 3, 'Water': 4,}
satellite_from_uav_classes = {'Seagrass': ['Seagrass', 'Seagrass submerged'],
 'Mixed': ['Gracilaria', 'Ulva', 'Cystophora', 'Hormosira', 'Gracilaria submerged', 'Submerged vegetation',
           'Brown algae mixed', 'Microphytobenthos', 'Green algae mixed', 
           'Filamentous brown algae', 'Ulva mats', 'Terrestrial', 'Red algae mixed'],
  'Unvegetated': ['Saltmarsh', 'Unvegetated', 'Rock', ],
                              
  'Water': ['Water'],
}

# Cells to run
* Train and save model
* Review model
  * satellite bands of each UAV class
  * satellite bands of each satellite class
  * importance of satellite bands in trained model
* Validation
  * Predict excluded site
  * Plot confusion matrix comparing prediction to UAV classifications

In [ ]:
cluster = dask.distributed.LocalCluster()
client = dask.distributed.Client(cluster)
display(client)

In [37]:
training_labels_file = data_path / "ELF24505_ClassificationClasses.txt"
sample_folder = utils.get_sample_folder()

test_satellite_file = data_path / "training" / "satellite_images" / f"{test_site}_sentinel-2.nc"
test_polygon_file = data_path / "site_polygons" / f"{test_site}_polygon.gpkg"
test_uav_file = data_path / "classified_orthos" / f"{test_site}_classified.tif"

{'Seagrass': 1, 'Mixed': 2, 'Unvegetated': 3, 'Water': 4}

### Train and save model

In [ ]:
# train and save the model
model, training_dataframe = utils_elf24505.train_classifier(
    data_path=data_path,
    training_sites=training_sites,
    training_path=training_path,
    uav_labels_file=uav_labels_file,
    uav_classes_to_ignore=uav_classes_to_ignore,
    satellite_classes=satellite_classes,
    satellite_from_uav_classes=satellite_from_uav_classes)

# Save model
joblib.dump(model, model_file); # , compress=3);

# Save a record of the classes considered in training
satellite_classes_dataframe = pandas.DataFrame.from_dict(satellite_classes, orient='index', columns=['satellite_class_id'])
satellite_classes_dataframe['uav_class_ids'] = satellite_classes_dataframe.index.map(lambda key: f"{satellite_from_uav_classes[key]}")
satellite_classes_dataframe.to_csv(model_file.with_name(f"{model_file.stem}_class_mappings.csv"), index=True)

print("Satellite training classes present: "
      f"{[key for key, value in satellite_classes.items() \
          if value in training_dataframe['satellite_class_id'].unique()]}")

### Review model

In [ ]:
# Plot satellite bands for UAV classes
number_uav_classes = len(training_dataframe["uav_class_id"].unique())
fig, axes = matplotlib.pyplot.subplots(nrows=int(numpy.ceil(number_uav_classes/3)), ncols=3, figsize=(14, 18))
for i, (class_id, ax) in enumerate(zip(training_dataframe["uav_class_id"].unique(), axes.flat)):
    training_dataframe[training_dataframe["uav_class_id"] == 1].drop(columns=["SCL", "uav_class_id", "satellite_class_id"]).boxplot(ax=ax)  
    ax.set_title(f"Spectral plot for class ID {class_id}")
matplotlib.pyplot.savefig(model_file.with_name(f"{model_file.stem}_training_uav_class_IDs.png"), dpi=300)

In [ ]:
# Plot satellite bands for the satellite class used for prediction
fig, axes = matplotlib.pyplot.subplots(nrows=2, ncols=2, figsize=(14, 12))
for i, (class_id, ax) in enumerate(zip(training_dataframe["satellite_class_id"].unique(), axes.flat)):
    training_dataframe[training_dataframe["satellite_class_id"] == 1].drop(columns=["SCL", "satellite_class_id", "uav_class_id"]).boxplot(ax=ax)  
    ax.set_title(f"Spectral plot for class ID {class_id}")
matplotlib.pyplot.savefig(model_file.with_name(f"{model_file.stem}_training_satellite_class_IDs.png"), dpi=300)

In [ ]:
# Plot the importance of each satellite band in the random forest model
importance_df = pandas.DataFrame(
    {'Feature': training_dataframe.drop(columns=["satellite_class_id", "uav_class_id", "time"]).columns,
     'Importance': model.feature_importances_})
importance_df.sort_values(by='Importance', ascending=False).plot(kind='bar', x='Feature', y='Importance');
matplotlib.pyplot.savefig(model_file.with_name(f"{model_file.stem}_random_forest_feature_importance.png"), dpi=300)

### Validate 

In [ ]:
predictions, satellite_data = utils_elf24505.predict_site(test_satellite_file=test_satellite_file, polygon_file=polygon_file, model_file=model_file)
utils_elf24505.save_netcdf(predictions, prediction_file)

print(
    f"Satellite training classes present: {[key for key, value in satellite_classes.items() if value in training_dataframe['satellite_class_id'].unique()]}. "
    f"Predicted classes present: {[key for key, value in satellite_classes.items() if value in numpy.unique(predictions.data)]}"
     )

In [ ]:
utils_elf24505.confusion_matrix_of_site(test_uav_file, uav_labels_file, prediction_file)